# Phase 2 KPI Validation

Manual validation notebook for `pm_spark.kpis.metrics` using `create_sample_event_log()`.

**Covered functions**
- `throughput_time()`
- `stp_rate()`
- `rework_rate()`
- `activity_frequency_distribution()`
- `case_volume_over_time()`
- `throughput_time_percentiles()`

**Implementation notes (current `metrics.py`)**
- Uniform public signatures: unused parameters are marked `# noqa: ARG001` and omitted from column validation (same idea as `eventlog/preparation.py`).
- `throughput_time` and `throughput_time_percentiles` both use `_per_case_duration()` for first→last span in the chosen `unit`.
- `activity_frequency_distribution` pulls the global event total with a one-row `collect()` and uses `F.when(total_count > 0, …)` so the plan never divides by zero.
- `case_volume_over_time(..., sort_result=True)` (default) runs `orderBy(period_start)`; set `sort_result=False` to skip that shuffle when downstream code sorts.
- `stp_rate` deduplicates `technical_users` before `isin()`.

In [1]:
from pathlib import Path
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Ensure project root is importable even when notebook cwd is dev/
project_root = Path.cwd().resolve()
if project_root.name == "dev":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

spark = SparkSession.getActiveSession()
if spark is None:
    spark = SparkSession.builder.appName("pm_spark_phase2_validation").getOrCreate()

from dev.sample_data import create_sample_event_log
from pm_spark.kpis.metrics import (
    activity_frequency_distribution,
    case_volume_over_time,
    rework_rate,
    stp_rate,
    throughput_time,
    throughput_time_percentiles,
)

def show_df(df, n=100):
    """Render Spark DataFrame output reliably across notebook runtimes."""

    df.show(n, truncate=False)

26/03/22 11:19:56 WARN Utils: Your hostname, MacBook-Air-von-Alex.local resolves to a loopback address: 127.0.0.1; using 192.168.178.137 instead (on interface en0)
26/03/22 11:19:56 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/22 11:19:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
event_log = create_sample_event_log(spark)
show_df(event_log.orderBy("case_key", "timestamp", "activity"))

+--------+-------------+-------------------+--------+----------+
|case_key|activity     |timestamp          |resource|department|
+--------+-------------+-------------------+--------+----------+
|C001    |A            |2026-03-18 09:00:00|R1      |D1        |
|C001    |B            |2026-03-18 09:05:00|R1      |D1        |
|C001    |C            |2026-03-18 09:10:00|R1      |D1        |
|C001    |D            |2026-03-18 09:20:00|R1      |D1        |
|C002    |A            |2026-03-18 09:00:00|R1      |D1        |
|C002    |B            |2026-03-18 09:04:00|R1      |D1        |
|C002    |Manual_Review|2026-03-18 09:08:00|R1      |D1        |
|C002    |C            |2026-03-18 09:15:00|R2      |D2        |
|C002    |D            |2026-03-18 09:25:00|R2      |D2        |
|C003    |A            |2026-03-18 10:00:00|NULL    |D1        |
|C003    |C            |2026-03-18 10:10:00|R3      |NULL      |
|C003    |D            |2026-03-18 10:30:00|R3      |D3        |
|C004    |A            |2

In [3]:
# 2.1 throughput_time()
# activity_col: uniform API only (not used or validated in this function).
df_throughput = throughput_time(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    unit="minutes",
    output_col="throughput_minutes",
)
show_df(df_throughput.orderBy("case_key"))

# Expected output:
# case_key="C001" -> throughput_minutes = 20.0 because first event is 09:00 and last is 09:20.
# case_key="C003" -> throughput_minutes = 30.0 because first event is 10:00 and last is 10:30.

+--------+------------------+
|case_key|throughput_minutes|
+--------+------------------+
|C001    |20.0              |
|C002    |25.0              |
|C003    |30.0              |
|C004    |30.0              |
|C005    |20.0              |
|C006    |30.0              |
|C007    |25.0              |
+--------+------------------+



In [4]:
# 2.2 stp_rate() with an actor column
# technical_users is deduplicated inside stp_rate() before isin().
# Build actor from resource and inject a human actor for manual review events.
event_log_with_actor = event_log.withColumn(
    "actor",
    F.when(F.col("activity") == "Manual_Review", F.lit("HUMAN_01"))
    .otherwise(F.coalesce(F.col("resource"), F.lit("SYSTEM"))),
)

technical_users = ["SYSTEM", "R1", "R2", "R3", "R4", "R5", "R6"]
df_stp = stp_rate(
    df=event_log_with_actor,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    actor_col="actor",
    technical_users=technical_users,
    null_strategy="exclude",
    output_col="is_stp",
)
show_df(df_stp.orderBy("case_key"))

# Expected output:
# case_key="C002" -> is_stp = False because activity 'Manual_Review' maps to actor 'HUMAN_01' (non-technical).
# case_key="C007" -> is_stp = True because null resources are mapped to actor 'SYSTEM' in this setup.

+--------+------+
|case_key|is_stp|
+--------+------+
|C001    |true  |
|C002    |false |
|C003    |true  |
|C004    |true  |
|C005    |true  |
|C006    |true  |
|C007    |true  |
+--------+------+



In [5]:
# 2.4 rework_rate()
# Single groupBy: count(non-null events) > size(collect_set(activity)); nulls excluded on both sides.
df_rework = rework_rate(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    output_col="has_rework",
)
show_df(df_rework.orderBy("case_key"))

# Expected output:
# case_key="C004" -> has_rework = True because activity 'B' appears twice.
# case_key="C001" -> has_rework = False because all activities are unique within the case.

+--------+----------+
|case_key|has_rework|
+--------+----------+
|C001    |false     |
|C002    |false     |
|C003    |false     |
|C004    |true      |
|C005    |false     |
|C006    |true      |
|C007    |false     |
+--------+----------+



In [6]:
# 2.5 activity_frequency_distribution()
# Global total is computed via one-row collect on the driver; relative frequency
# is null when total_count == 0 (defensive), otherwise count / total.
df_activity_freq = activity_frequency_distribution(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    count_col="activity_count",
    frequency_col="activity_relative_frequency",
)
show_df(df_activity_freq.orderBy("activity"))

# Expected output:
# activity="A" -> activity_count = 8 because it appears in all 7 cases plus one extra in loop case C006.
# activity="Manual_Review" -> activity_count = 1 because it appears only once in case C002.

+-------------+--------------+---------------------------+
|activity     |activity_count|activity_relative_frequency|
+-------------+--------------+---------------------------+
|A            |8             |0.2581                     |
|B            |7             |0.2258                     |
|C            |7             |0.2258                     |
|D            |7             |0.2258                     |
|Manual_Review|1             |0.0323                     |
|P            |1             |0.0323                     |
+-------------+--------------+---------------------------+



In [7]:
# 2.6 case_volume_over_time()
# sort_result=True (default): Spark orderBy(period_start). False skips it if you sort later.
df_case_volume_day = case_volume_over_time(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    frequency="day",
    output_col="case_volume",
    sort_result=True,
)
show_df(df_case_volume_day)

df_case_volume_week = case_volume_over_time(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    frequency="week",
    output_col="case_volume",
    sort_result=False,
)
# No global sort from the function — order here only for readable output.
show_df(df_case_volume_week.orderBy("period_start"))

# Expected output:
# period_start=2026-03-18 (day) -> case_volume = 4 for C001, C002, C003, C004.
# period_start=2026-03-19 (day) -> case_volume = 3 for C005, C006, C007.

+-------------------+-----------+
|period_start       |case_volume|
+-------------------+-----------+
|2026-03-18 00:00:00|4          |
|2026-03-19 00:00:00|3          |
+-------------------+-----------+

+-------------------+-----------+
|period_start       |case_volume|
+-------------------+-----------+
|2026-03-16 00:00:00|7          |
+-------------------+-----------+



In [8]:
# 2.7 throughput_time_percentiles()
# Per-case durations use the same _per_case_duration() path as throughput_time().
df_percentiles = throughput_time_percentiles(
    df=event_log,
    case_col="case_key",
    activity_col="activity",
    timestamp_col="timestamp",
    unit="minutes",
    accuracy=10000,
    p50_col="p50_minutes",
    p75_col="p75_minutes",
    p90_col="p90_minutes",
    p95_col="p95_minutes",
)
show_df(df_percentiles)

# Expected output:
# p50_minutes should be around 25.0 because sorted throughputs are [20, 25, 25, 30, 30, 30, 30].
# p95_minutes should be around 30.0 because the upper tail is dominated by 30-minute cases.

+-----------+-----------+-----------+-----------+
|p50_minutes|p75_minutes|p90_minutes|p95_minutes|
+-----------+-----------+-----------+-----------+
|25.0       |30.0       |30.0       |30.0       |
+-----------+-----------+-----------+-----------+



In [9]:
# Consolidated view for quick sanity checks
df_kpi_case = (
    df_throughput
    .join(df_stp, on="case_key", how="left")
    .join(df_rework, on="case_key", how="left")
)
show_df(df_kpi_case.orderBy("case_key"))

+--------+------------------+------+----------+
|case_key|throughput_minutes|is_stp|has_rework|
+--------+------------------+------+----------+
|C001    |20.0              |true  |false     |
|C002    |25.0              |false |false     |
|C003    |30.0              |true  |false     |
|C004    |30.0              |true  |true      |
|C005    |20.0              |true  |false     |
|C006    |30.0              |true  |true      |
|C007    |25.0              |true  |false     |
+--------+------------------+------+----------+



## Validation Checklist (Phase 2)

Mark each item after you run the corresponding cell and confirm outputs (see implementation notes at the top).

- [ ] 2.1 `throughput_time()` validated
- [ ] 2.2 `stp_rate()` validated
- [ ] 2.3 `dunkelquote()` removed and covered by `stp_rate()`
- [ ] 2.4 `rework_rate()` validated
- [ ] 2.5 `activity_frequency_distribution()` validated
- [ ] 2.6 `case_volume_over_time()` validated
- [ ] 2.7 `throughput_time_percentiles()` validated

When all boxes are checked, update `PROGRESS.md` Phase 2 entries to `[x]`.